# Visualising the trajectory with py3Dmol

This renders the production trajectory **inline in the notebook** using
`py3Dmol`. Unlike `nglview`, py3Dmol injects the 3Dmol.js viewer directly into
the cell output and does **not** depend on the JupyterLab widget system, so it
works on this Hub where nglview does not.

Inputs (created by the `analysis` notebook, in this `analysis/` folder):

| File | Role |
| ---- | ---- |
| `frame0.pdb` | topology: atoms + bonds (one reference frame) |
| `prod_noW.xtc` | trajectory: how the system moves over time |

> **What happens here:** MDAnalysis reads the topology and trajectory, we write a
> few frames into a single multi-frame PDB, and py3Dmol plays them as an animation.
> The protein domains are coloured (Ig domain, hinge, TM helix) and the POPC
> membrane is drawn as translucent sticks.

## Step 1 — Setup and load the trajectory

In [ ]:
# Ensure py3Dmol is importable (auto-install into your own home dir if missing).
import sys, os, subprocess, importlib
_t = os.path.expanduser('~/.local/py3dmol')
if _t not in sys.path:
    sys.path.insert(0, _t)
try:
    import py3Dmol
except ModuleNotFoundError:
    print('Installing py3Dmol into', _t, '(one-time)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', _t, 'py3Dmol'], check=False)
    importlib.invalidate_caches()
    import py3Dmol
# py3Dmol was installed into your home dir (missing from the shared venv).
import sys, os

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import MDAnalysis as mda
from MDAnalysis.coordinates.PDB import PDBWriter
import tempfile
print('py3Dmol', py3Dmol.__version__, '| MDAnalysis', mda.__version__)

In [ ]:
# Load topology (frame0.pdb) + trajectory (prod_noW.xtc) from this folder.
topology_file   = 'frame0.pdb'
trajectory_file = 'prod_noW.xtc'

for f in (topology_file, trajectory_file):
    assert os.path.exists(f), f'Missing {f} - run the trajectory-prep cell in the analysis notebook first'

u = mda.Universe(topology_file, trajectory_file)
print('atoms/beads:', len(u.atoms))
print('residues   :', len(u.residues), f'(resid {u.residues.resids.min()}-{u.residues.resids.max()})')
print('frames     :', len(u.trajectory))

## Step 2 — Define the domains to colour

> **What happens here:** we pick residue ranges for the soluble Ig domain, the
> hinge, the transmembrane (TM) helix, and the rest. **Adjust these to your
> protein** using the residue range printed above. The defaults come from the
> original tutorial's protein (CADM1) and may not match yours.

Martini has no atomistic backbone, so beads are coloured by residue id rather
than by `protein` selection.

In [ ]:
# Residue ranges per region (EDIT to match your protein).
ig_range    = '3:89'      # soluble Ig domain
hinge_range = '90:107'    # hinge / linker
tm_range    = '108:128'   # transmembrane helix

# Membrane lipids (Martini POPC). If your lipid name differs, change it here.
popc = u.select_atoms('resname POPC')
print('POPC beads:', len(popc))

# 0-based atom indices for each region (py3Dmol serial = index+1 in the PDB we write).
ig_idx    = u.select_atoms(f'resid {ig_range}').indices
hinge_idx = u.select_atoms(f'resid {hinge_range}').indices
tm_idx    = u.select_atoms(f'resid {tm_range}').indices
popc_idx  = popc.indices
print('Ig:', len(ig_idx), ' hinge:', len(hinge_idx), ' TM:', len(tm_idx))

## Step 3 — Animate the trajectory

> **What happens here:** we write up to `n_frames` snapshots into one multi-frame
> PDB and hand it to py3Dmol. Each region is styled by atom serial: Ig (crimson),
> hinge (orange), TM helix (navy), membrane (silver, translucent). Press play in
> the viewer to watch the protein move in the bilayer.

In [ ]:
# Ensure py3Dmol is importable (auto-install into your own home dir if missing).
import sys, os, subprocess, importlib
_t = os.path.expanduser('~/.local/py3dmol')
if _t not in sys.path:
    sys.path.insert(0, _t)
try:
    import py3Dmol
except ModuleNotFoundError:
    print('Installing py3Dmol into', _t, '(one-time)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', _t, 'py3Dmol'], check=False)
    importlib.invalidate_caches()
    import py3Dmol
# ===== Smooth animated trajectory (py3Dmol) =====
# The trajectory may have only a few saved frames (e.g. 1 ns apart). To make the
# animation fluid, we LINEARLY INTERPOLATE extra frames between the saved ones in
# Python, then play them all with a short interval. Rotating does not reset the view.
import sys, os, tempfile
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import MDAnalysis as mda
from MDAnalysis.coordinates.PDB import PDBWriter
from MDAnalysis.coordinates.memory import MemoryReader

topo = 'frame0.pdb'
traj = 'prod_noW.xtc'
assert os.path.exists(topo) and os.path.exists(traj), 'run the trajectory-prep cell first'

u = mda.Universe(topo, traj)
n_saved = len(u.trajectory)
times_ns = [ts.time / 1000.0 for ts in u.trajectory]
print(f'{{n_saved}} saved frames.  frame -> time:')
for i, t in enumerate(times_ns):
    print(f'  frame {{i}}: {{t:.2f}} ns')

# Collect saved-frame coordinates.
coords = np.array([u.trajectory[i].positions.copy() for i in range(n_saved)])

# Interpolate: insert SUB-1 frames between each pair of saved frames.
SUB = 10  # interpolation factor; higher = smoother but heavier
if n_saved >= 2:
    interp = []
    for a in range(n_saved - 1):
        for k in range(SUB):
            f = k / SUB
            interp.append((1 - f) * coords[a] + f * coords[a + 1])
    interp.append(coords[-1])
    coords_interp = np.array(interp)
else:
    coords_interp = coords
print(f'{{len(coords_interp)}} frames after interpolation (SUB={{SUB}})')

# Build a Universe from the interpolated coordinates and write a multi-frame PDB.
u2 = mda.Universe(topo)
u2.load_new(coords_interp, format=MemoryReader)
with tempfile.NamedTemporaryFile(suffix='.pdb', delete=False) as tmp:
    tmp_path = tmp.name
with PDBWriter(tmp_path, multiframe=True) as W:
    for ts in u2.trajectory:
        W.write(u2.atoms)
with open(tmp_path) as fh:
    pdb_data = fh.read()
os.unlink(tmp_path)

view = py3Dmol.view(width=900, height=650)
view.addModelsAsFrames(pdb_data, 'pdb')
def serials(idx):
    return {'serial': [int(j) + 1 for j in idx]}
view.setStyle({}, {'line': {'color': 'lightgrey'}})
view.setStyle(serials(ig_idx),    {'sphere': {'color': 'crimson', 'radius': 1.6}})
view.setStyle(serials(hinge_idx), {'sphere': {'color': 'orange',  'radius': 1.6}})
view.setStyle(serials(tm_idx),    {'sphere': {'color': 'navy',    'radius': 1.6}})
if len(popc_idx):
    view.setStyle(serials(popc_idx), {'stick': {'color': 'silver', 'radius': 0.15, 'opacity': 0.35}})
view.zoomTo()
view.animate({'loop': 'forward', 'interval': 50, 'reps': 0, 'step': 1})
view.setBackgroundColor('white')
view.show()


## Step 4 — Reading the view

- **Crimson** = Ig domain, **orange** = hinge, **navy** = TM helix, **silver** =
  POPC membrane.
- Drag to rotate. Turn the membrane side-on (edge of the silver slab facing you)
  to see the protein crossing the bilayer, Ig domain above, TM helix inside.
- The playback loops automatically; the protein and glycan/loop motions show the
  dynamics sampled during production.

> If a region looks empty, its residue range probably does not match your protein
> — edit the ranges in Step 2 and re-run.